# Kaggle full source download test

CPU-only preflight on the established notebook9 identity. It attempts all LADI-v2 data files and the complete official MEDIC and CrisisMMD archives, writes only under `/kaggle/working/experiments/dataset_detector_grid/runs/<UTC run id>/`, and maintains a 3 GB free-space reserve. It does not extract archives, train models, or retrieve image archives to the laptop.

The existing private input dataset remains attached for identity continuity; this test downloads from the published source URLs.


In [ ]:
# Setup, provenance, and bounded runtime directory
import hashlib, json, os, platform, shutil, sys, time, traceback
from datetime import datetime, timezone
from pathlib import Path

SOURCE_TEXT = "\"\"\"Cloud-only full-source download attempt with disk, progress and receipts.\"\"\"\nimport hashlib\nimport json\nimport shutil\nimport time\nfrom pathlib import Path\nimport requests\n\ndef save(path, value):\n    tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(value,indent=2));tmp.replace(path)\n\ndef run(root, seconds=3600):\n    root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    start=time.monotonic();results=[]\n    from huggingface_hub import HfApi, hf_hub_url\n    api=HfApi();info=api.dataset_info('MITLL/LADI-v2-dataset',files_metadata=True)\n    sources=[{'source':'CrisisMMD','name':'CrisisMMD_v2.0.tar.gz','url':'https://crisisnlp.qcri.org/data/crisismmd/CrisisMMD_v2.0.tar.gz','expected_bytes':None},\n             {'source':'MEDIC','name':'MEDIC.tar.gz','url':'https://crisisnlp.qcri.org/data/MEDIC/MEDIC.tar.gz','expected_bytes':None}]\n    for f in info.siblings:\n        if f.rfilename.startswith('data/'):\n            sources.append({'source':'LADI-v2','name':f.rfilename,'url':hf_hub_url(info.id,f.rfilename,repo_type='dataset',revision=info.sha),'expected_bytes':f.size})\n    save(root/'inventory.json',{'ladi_revision':info.sha,'sources':sources,'scope':'all listed LADI data files plus official complete archives; no extraction'})\n    try:\n        for entry in sources:\n            if time.monotonic()-start>=seconds:\n                results.append({**entry,'status':'time_budget_exhausted'});break\n            dest=root/'downloads'/entry['source']/entry['name'];dest.parent.mkdir(parents=True,exist_ok=True)\n            row={**entry,'status':'running','bytes':0};results.append(row);save(root/'downloads.json',results)\n            print('DOWNLOAD START',entry['source'],entry['name'],flush=True)\n            try:\n                with requests.get(entry['url'],stream=True,timeout=(30,90)) as response:\n                    response.raise_for_status()\n                    expected=entry['expected_bytes'] or int(response.headers.get('Content-Length',0)) or None\n                    free=shutil.disk_usage(root).free;row.update(expected_bytes=expected,free_bytes_before=free)\n                    if expected and expected+3_000_000_000>free:\n                        row['status']='insufficient_disk';save(root/'downloads.json',results);continue\n                    digest=hashlib.sha256();last=time.monotonic();part=dest.with_suffix(dest.suffix+'.part')\n                    with part.open('wb') as out:\n                        for chunk in response.iter_content(4*1024*1024):\n                            if not chunk:continue\n                            if time.monotonic()-start>=seconds:raise TimeoutError('download execution budget reached')\n                            if shutil.disk_usage(root).free<len(chunk)+3_000_000_000:raise OSError('3GB disk reserve reached')\n                            out.write(chunk);digest.update(chunk);row['bytes']+=len(chunk)\n                            if time.monotonic()-last>=15:\n                                save(root/'downloads.json',results);print('DOWNLOAD PROGRESS',entry['source'],row['bytes'],expected,flush=True);last=time.monotonic()\n                    if expected and row['bytes']!=expected:raise ValueError('download length mismatch')\n                    if response.headers.get('Content-Type','').startswith('text/html'):raise ValueError('received HTML instead of dataset')\n                    part.replace(dest);row.update(status='complete',sha256=digest.hexdigest(),path=str(dest))\n            except Exception as error:\n                row.update(status='failed',error=type(error).__name__,message=str(error))\n            save(root/'downloads.json',results);print('DOWNLOAD END',row['status'],row['bytes'],flush=True)\n    finally:\n        save(root/'summary.json',{'seconds':time.monotonic()-start,'complete_files':sum(r['status']=='complete' for r in results),\n             'planned_files':len(sources),'all_complete':len(results)==len(sources) and all(r['status']=='complete' for r in results),\n             'downloaded_bytes':sum(r['bytes'] for r in results if 'bytes' in r),'free_bytes':shutil.disk_usage(root).free,\n             'persistence':'Kaggle working files; durable notebook output must be verified separately'})\n    return results\n"
EXPECTED_SOURCE_SHA256 = '06f19ff2598c0de8b9d865923c933db6be82836d554449c3302ea209a79e6e30'
actual_source_sha256 = hashlib.sha256(SOURCE_TEXT.encode("utf-8")).hexdigest()
assert actual_source_sha256 == EXPECTED_SOURCE_SHA256, "embedded download source hash mismatch"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_dataset_download"
run_root = Path("/kaggle/working/experiments/dataset_detector_grid/runs") / run_id
run_root.mkdir(parents=True, exist_ok=False)
log_path = run_root / "stdout.log"
class Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, value):
        for stream in self.streams: stream.write(value); stream.flush()
        return len(value)
    def flush(self):
        for stream in self.streams: stream.flush()
    def isatty(self): return False
    @property
    def encoding(self): return getattr(self.streams[0], "encoding", "utf-8")

log_file = log_path.open("a", encoding="utf-8", buffering=1)
_original_stdout, _original_stderr = sys.stdout, sys.stderr
sys.stdout = Tee(_original_stdout, log_file)
sys.stderr = Tee(_original_stderr, log_file)
disk = shutil.disk_usage("/kaggle/working")
print("PREFLIGHT START", run_id, flush=True)
print(json.dumps({"python_executable": sys.executable, "python_version": sys.version, "platform": platform.platform(), "requests": __import__("requests").__version__, "run_root": str(run_root), "source_sha256": actual_source_sha256, "free_bytes_before": disk.free, "reserve_bytes": 3000000000, "download_budget_seconds": 3600}, sort_keys=True), flush=True)
(run_root / "provenance.json").write_text(json.dumps({"run_id": run_id, "source_sha256": actual_source_sha256, "python_executable": sys.executable, "python_version": sys.version, "platform": platform.platform(), "free_bytes_before": disk.free, "reserve_bytes": 3000000000, "download_budget_seconds": 3600, "notebook_ref": "satyapaladugu/betterscheduler-spi-20261027finalc", "gpu_enabled": False}, indent=2), encoding="utf-8")


In [ ]:
# Execute the exact committed source with incremental logging and failure receipt
import json, shutil, time, traceback
run_started = time.monotonic()
reserve = 3_000_000_000
free_before = shutil.disk_usage("/kaggle/working").free
if free_before < reserve:
    result = {"status": "insufficient_initial_disk", "free_bytes_before": free_before, "reserve_bytes": reserve}
    (run_root / "summary.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    print("DOWNLOAD SKIPPED: initial free space is below the required reserve", flush=True)
else:
    try:
        namespace = {"__name__": "kaggle_dataset_download_test"}
        exec(compile(SOURCE_TEXT, "download_test.py", "exec"), namespace)
        namespace["run"](run_root, seconds=3600)
        print("DOWNLOAD TEST FUNCTION COMPLETED", flush=True)
    except BaseException as error:
        failure = {"status": "runner_exception", "error_type": type(error).__name__, "message": str(error), "traceback": traceback.format_exc(), "elapsed_seconds": time.monotonic() - run_started, "free_bytes": shutil.disk_usage("/kaggle/working").free}
        (run_root / "runner_failure.json").write_text(json.dumps(failure, indent=2), encoding="utf-8")
        print("DOWNLOAD TEST FAILED; traceback saved", flush=True)
        traceback.print_exc()
free_after = shutil.disk_usage("/kaggle/working").free
run_summary = {"elapsed_seconds": time.monotonic() - run_started, "free_bytes_after": free_after, "reserve_bytes": reserve, "reserve_preserved": free_after >= reserve, "run_root": str(run_root)}
(run_root / "notebook_summary.json").write_text(json.dumps(run_summary, indent=2), encoding="utf-8")
print("DOWNLOAD TEST CELL COMPLETE", json.dumps(run_summary, sort_keys=True), flush=True)
